### The inventory baseline 

The inventory baseline establishes the **Ideal state** of the inventory data.

It tracks the movement of inventory for the simulation timeline which is December 1st, 2025 to June 30th, 2026. 

Generates 365600+ events across 211 days for 100 darkstores. 

Event types are: snapshots, fulfillment and RTOs, and transfers

Key logic:
- Demand spikes applied from config.yaml 
- RTO rates vary by product category (dairy 8%, staples 2%)
- Replenishment chain: darkstore > regional hub > mother hub escalation
- Transfer window enforced at 03:00-05:00 before morning snapshot
- Temporal gate skips darkstores before their operational_since date

**Schema example**
___
| event_id | event_type | event_timestamp | node_id | device_id | product_id | units_sold | units_rto | units_transferred | source_node | destination_node | stock_on_hand | rto_reason | processing_lag_hours | session_id |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |:--- |:--- |:--- |:--- |:--- |:--- |:--- |
| EVT_0365635 | snapshot | 2026-06-30 23:30:00 | DS_NGP_02 | SCAN_NGP_001 |  | 0 | 0 | 0 |  |  | 421 |  | 0.899 | SES_DS_NGP_02_20260630_SNAP_2330 |
___

In [ ]:
import yaml
import random
import numpy as np
import pandas as pd
from datetime import date, datetime, timedelta
from validate_config import validate_config

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

validate_config(cfg)

random.seed(cfg["simulation"]["random_seed"])
np.random.seed(cfg["simulation"]["random_seed"])

SIM_START = date.fromisoformat(cfg["simulation"]["start_date"])
SIM_END = date.fromisoformat(cfg["simulation"]["end_date"])

## loading the master data
df_supplier = pd.read_csv("dim_supplier.csv")
df_product = pd.read_csv("dim_product.csv")
df_node = pd.read_csv("dim_node.csv")
df_hierarchy = pd.read_csv("dim_hub_hierarchy.csv")
df_scanner = pd.read_csv("dim_scanner.csv")

## lookups for the master data
node_lookup = df_node.set_index("node_id").to_dict("index")
scanner_lookup = df_scanner.groupby("assigned_node_id")["device_id"].apply(list).to_dict()
product_lookup = df_product.set_index("product_id").to_dict("index")
hierarchy_lookup = df_hierarchy.set_index("darkstore_id").to_dict("index")

darkstores = df_node[df_node["node_type"] == "darkstore"].copy()
regional_hubs = df_node[df_node["node_type"] == "regional_hub"].copy()
mother_hubs = df_node[df_node["node_type"] == "mother_hub"].copy()

print(f"Darkstores: {len(darkstores)}")
print(f"Products: {len(df_product)}")
print(f"Sim days: {(SIM_END - SIM_START).days}")

Darkstores: 100
Products: 100
Sim days: 211


event_type options:
- snapshot (stock count at scheduled times, 6:00, 13:00, 18:00, 23:30)
- inbound_transfer (hub sends stock to darkstore)
- lateral_transfer (darkstore sends stick to sibling darkstore)
- customer_fulfillment (units sold, reduces stock)
- rto_return (units returned, increases the stock)

In [2]:
event_schema ={
    "event_id": str, # evt_00001 and so on
    "event_type": str, # one of the 5 event type options
    "event_timestamp": datetime,
    "node_id": str, # the node at which the event is occuring
    "device_id": str, # id of the scanner logging the movement
    "product_id": str,
    "units_sold": int, # value>0 for fulfillment, otherwise 0
    "units_rto": int, # value > 0 for return, otherwise 0
    "units_transferred": int, # value >0 for transfer events, otherwise 0
    "source_node": str, # for transfers where the stock came from
    "destination_node": str, # for transfers, where the stock went
    "stock_on_hand": int, # current stock AFTER this event
    "rto_reason": str, # nullable, only str when rto_return is there
    "processing_lag_hours": float, #normal 0-4, chaos will inject outliers
    "session_id": str, # groups scanner pings from one session
}

In [3]:
# helper fuctions:
def get_demand_multiplier(current_date, zone_type, node_tier, category, city):
    """
    Returns the demand multiplier for this node/category/date combination.
    Reads demand_spike_events from config.yaml
    Returns 1.0 if no spikes
    """
    best = 1.0
    for spike in cfg["demand_spike_events"]:
        # date match
        if "date" in spike:
            if current_date != date.fromisoformat(str(spike["date"])):
                continue
        elif "date_range" in spike:
            start = date.fromisoformat(spike["date_range"][0])
            end = date.fromisoformat(spike["date_range"][1])
            if not (start <= current_date <= end):
                continue
        else:
            continue
                
        #category match
        cats = spike["product_category"]
        if isinstance(cats, str):
            cats = [cats]
        if category not in cats:
            continue
                
        # scope match
        if "scope" in spike:
            if spike["scope"] == "metropolitan" and node_tier != "metropolitan":
                continue
            if "cities" in spike and city not in spike["cities"]:
                continue

        # zone_type match
        if "zone_type" in spike:
            if zone_type != spike["zone_type"]:
                continue

        best = max(best, spike["multiplier"])

    return best
        

def pick_scanner(node_id, zone_pref=None):
    """
    Returns a device_id for the given node, preferring zone_pref.
    """
    node_scanners = df_scanner[df_scanner["assigned_node_id"] == node_id]
    if zone_pref:
        preferred = node_scanners[node_scanners["scanner_zone"] == zone_pref]
        if len(preferred) > 0:
            return random.choice(preferred["device_id"].tolist())
    if len(node_scanners) > 0:
        return random.choice(node_scanners["device_id"].tolist())
    return f"SCAN_FALLBACK_{node_id}"

def session_id(node_id, current_date, tag):
    """Groups scanner pings from one session together"""
    return f"SES_{node_id}_{current_date.strftime('%Y%m%d')}_{tag}"
        
def base_daily_demand(node_capacity, category, node_tier):
    """
    Approximate units of one product sold per day at this node.
    Perishables turn over faster. Metro nodes move more volume.
    """
    turnover = {
        "dairy": 0.007,
        "beverages": 0.005,
        "staples": 0.003,
        "snacks": 0.004,
        "personal_care": 0.002,
        "household": 0.002,
        "frozen": 0.003,
    }
    tier_scale = {"metropolitan": 1.0, "tier_1": 0.65, "tier_2": 0.35}

    base = (
        node_capacity * turnover.get(category, 0.003) * tier_scale.get(node_tier, 0.5)
    )
    return max(1, int(base))
        
# product lookup for fast access inside loops
prod_cfg = cfg["product_categories"]
product_list = df_product.to_dict("records") # list of dicts, faster than iterrows

RTO_REASONS = [
    "Customer not available",
    "Damaged in transit",
    "Wrong item delivered",
    "Quality issue on arrival",
    "Customer refused delivery",
]

print("Helper functions ready.")

Helper functions ready.


### **SnapShot events**

In [4]:
stock_state = {}
for _, node in df_node.iterrows():
    stock_state[node["node_id"]] = {}
    for _, prod in df_product.iterrows():
        reorder_pt = prod["reorder_point_units"]
        if node["node_type"] == "darkstore":
            per_product = random.randint(reorder_pt * 2, reorder_pt * 4)
        elif node["node_type"] == "regional_hub":
            per_product = random.randint(reorder_pt * 10, reorder_pt * 20)
        else:
            per_product = random.randint(reorder_pt * 50, reorder_pt * 100)
        stock_state[node["node_id"]][prod["product_id"]] = per_product

events = []
event_counter = 1
current_date = SIM_START
total_days = (SIM_END - SIM_START).days

print(f"Generating {total_days} days of discrete inventory events...")
print(f"Expected rows: ~{total_days * len(darkstores)*20:,}")

def generate_snapshots(current_date, node_id, node_type, stock_state, event_counter):
    """
    Generates stock per count snapshots at scheduled times.
    One snapshot row per scheduled time per node (captures total stock). 
    """
    snaps = []
    schedule_key = {
        "darkstore": "darkstore",
        "regional_hub": "regional_hub",
        "mother_hub": "mother_hub",
    }.get(node_type, "darkstore")

    times = cfg["snapshot_schedule"][schedule_key]

    for time_str in times:
        hour, minute = map(int, time_str.split(":"))
        ts = datetime.combine(
            current_date,
            datetime.min.time().replace(hour = hour, minute = minute)
        )

        total_stock = sum(stock_state[node_id].values())
        snaps.append({
            "event_id": f"EVT_{event_counter:07d}",
            "event_type": "snapshot",
            "event_timestamp": ts,
            "node_id": node_id,
            "device_id": pick_scanner(node_id, "stock_count"),
            "product_id": None,
            "units_sold": 0,
            "units_rto": 0,
            "units_transferred": 0,
            "source_node": None,
            "destination_node": None,
            "stock_on_hand": total_stock,
            "rto_reason": None,
            "processing_lag_hours": round(random.uniform(0.0, 2.0), 3),
            "session_id": session_id(node_id, current_date, f"SNAP_{time_str.replace(':','')}"),

        })
        event_counter += 1


    return  snaps, event_counter
test_snaps, _ = generate_snapshots(SIM_START, "DS_AND_01", "darkstore", stock_state,1)
print(f"Snapshots for one node on one day: {len(test_snaps)}")
for s in test_snaps:
    print(f" {s['event_timestamp']} total_stock={s['stock_on_hand']}")

Generating 211 days of discrete inventory events...
Expected rows: ~422,000
Snapshots for one node on one day: 4
 2025-12-01 06:00:00 total_stock=14744
 2025-12-01 13:00:00 total_stock=14744
 2025-12-01 18:00:00 total_stock=14744
 2025-12-01 23:30:00 total_stock=14744


### **Fulfillments and RTOs**

In [5]:
def generate_fulfillments_and_rtos(current_date, ds_row, stock_state, event_counter):
    """
    For each active product today:
      - generates one fulfillment event (daily aggregated demand)
      - possibly generates one RTO event (based on category rto_rate_base)
    """
    node_id  = ds_row["node_id"]
    capacity = ds_row["capacity_units"]
    tier= ds_row["tier"]
    zone_type = ds_row["zone_type"]
    city = ds_row["city"]
    day_events = []

    category_weights = {
        "dairy": 3, "beverages": 3, "snacks": 3,
        "staples": 2, "frozen": 2,
        "personal_care": 1, "household": 1,
    }
    weights = [category_weights.get(p["category"], 1) for p in product_list]
    n_active = random.randint(10, 20)
    active_products = random.choices(product_list, weights=weights, k=n_active)

    for prod in active_products:
        prod_id  = prod["product_id"]
        category = prod["category"]
        current_stock = stock_state[node_id][prod_id]

        if current_stock <= 0:
            continue

        multiplier = get_demand_multiplier(current_date, zone_type, tier, category, city)
        base  = base_daily_demand(capacity, category, tier)
        noise = random.uniform(0.6, 1.4)
        units = min(int(base * multiplier * noise), current_stock)

        if units <= 0:
            continue

        stock_state[node_id][prod_id] -= units

        ts = datetime.combine(
            current_date,
            datetime.min.time().replace(
                hour=random.randint(7, 23),
                minute=random.randint(0, 59)
            )
        )

        day_events.append({
            "event_id":             f"EVT_{event_counter:07d}",
            "event_type":           "customer_fulfillment",
            "event_timestamp":      ts,
            "node_id":              node_id,
            "device_id":            pick_scanner(node_id, "outbound"),
            "product_id":           prod_id,
            "units_sold":           units,
            "units_rto":            0,
            "units_transferred":    0,
            "source_node":          None,
            "destination_node":     None,
            "stock_on_hand":        stock_state[node_id][prod_id],
            "rto_reason":           None,
            "processing_lag_hours": round(random.uniform(0.0, 2.0), 3),
            "session_id":           session_id(node_id, current_date, "FUL"),
        })
        event_counter += 1

        # RTO check
        rto_rate = prod_cfg[category]["rto_rate_base"]
        if random.random() < rto_rate:
            rto_qty = max(1, int(units * random.uniform(0.05, 0.20)))
            stock_state[node_id][prod_id] += rto_qty
            rto_ts = ts + timedelta(hours=random.randint(1, 6))

            day_events.append({
                "event_id":             f"EVT_{event_counter:07d}",
                "event_type":           "rto_return",
                "event_timestamp":      rto_ts,
                "node_id":              node_id,
                "device_id":            pick_scanner(node_id, "returns"),
                "product_id":           prod_id,
                "units_sold":           0,
                "units_rto":            rto_qty,
                "units_transferred":    0,
                "source_node":          None,
                "destination_node":     None,
                "stock_on_hand":        stock_state[node_id][prod_id],
                "rto_reason":           random.choice(RTO_REASONS),
                "processing_lag_hours": round(random.uniform(0.0, 1.0), 3),
                "session_id":           session_id(node_id, current_date, "RTO"),
            })
            event_counter += 1

    return day_events, event_counter, stock_state

print("Fulfillment + RTO function ready.")

Fulfillment + RTO function ready.


### **Generate Transfers**

In [6]:
rep_cfg = cfg["replenishment"]

def generate_transfers(current_date, node_id, stock_state, event_counter):
    """
    Checks each product at this darkstore.
    If stock < reorder_point, requests transfer from parent regional hub.
    If regional hub is also low, escalates to mother hub.
    Transfer timestamp is set to overnight window (03:00-05:00).
    """
    if node_id not in hierarchy_lookup:
        return [], event_counter, stock_state

    rh_id = hierarchy_lookup[node_id]["regional_hub_id"]
    mh_id = hierarchy_lookup[node_id]["mother_hub_id"]
    top_up = rep_cfg["top_up_target_fraction"]
    transfers = []
    node_capacity = node_lookup[node_id]["capacity_units"]

    for prod in product_list:
        prod_id = prod["product_id"]
        reorder_pt = prod["reorder_point_units"]
        current_stock = stock_state[node_id][prod_id]

        if current_stock >= reorder_pt:
            continue

        target = int(node_capacity* top_up/ len(product_list))
        needed = max(0, target - current_stock)

        if needed <= 0:
            continue

        rh_stock = stock_state.get(rh_id, {}).get(prod_id, 0)
        source = None
        qty = 0

        if rh_stock >= needed:
            source = rh_id
            qty = needed
            stock_state[rh_id][prod_id] -= qty
        elif rh_stock > 0:
            source = rh_id
            qty = rh_stock
            stock_state[rh_id][prod_id] = 0
        else:
            mh_stock = stock_state.get(mh_id, {}).get(prod_id, 0)
            if mh_stock >= needed:
                source = mh_id
                qty = needed
                stock_state[mh_id][prod_id] -= qty

        if source and qty > 0:
            stock_state[node_id][prod_id] += qty
            transfer_ts = datetime.combine(
                current_date,
                datetime.min.time().replace(
                    hour=random.randint(3, 5),
                    minute=random.randint(0, 59)
                )
            )
            transfers.append({
                "event_id": f"EVT_{event_counter:07d}",
                "event_type": "inbound_transfer",
                "event_timestamp": transfer_ts,
                "node_id": node_id,
                "device_id": pick_scanner(node_id, "inbound"),
                "product_id": prod_id,
                "units_sold": 0,
                "units_rto": 0,
                "units_transferred": qty,
                "source_node": source,
                "destination_node": node_id,
                "stock_on_hand":stock_state[node_id][prod_id],
                "rto_reason": None,
                "processing_lag_hours": round(random.uniform(0.5, 3.0), 3),
                "session_id": session_id(node_id, current_date, "TRF"),
            })
            event_counter += 1

    return transfers, event_counter, stock_state

print("Transfer function ready.")

Transfer function ready.


### **Generate Lateral Transfers**

In [7]:
def generate_lateral_transfers(current_date, node_id, stock_state, event_counter):
    """
    Darkstore-to-darkstore lateral transfer.
    Triggered when a darkstore has surplus stock (> 80% capacity for a product)
    AND a sibling darkstore under the same regional hub is below reorder point.
    Transfer window: 10:00-16:00 (off-peak, per config).
    """
    if node_id not in hierarchy_lookup:
        return [], event_counter, stock_state

    rh_id = hierarchy_lookup[node_id]["regional_hub_id"]
    min_surplus = rep_cfg["lateral_transfer_min_surplus"]
    transfer_frac = rep_cfg["lateral_transfer_fraction"]

    # find sibling darkstores under same regional hub
    siblings = [
        ds_id for ds_id, meta in hierarchy_lookup.items()
        if meta["regional_hub_id"] == rh_id and ds_id != node_id
    ]

    if not siblings:
        return [], event_counter, stock_state

    transfers = []
    node_capacity = node_lookup[node_id]["capacity_units"]
    per_product_capacity = int(node_capacity * 0.80 / len(product_list))

    for prod in product_list:
        prod_id = prod["product_id"]
        reorder_pt = prod["reorder_point_units"]
        source_stock = stock_state[node_id][prod_id]

    # surplus = anything above 2x the reorder point
    # realistic: a darkstore with 80 units when reorder is 30 has genuine surplus
        surplus = source_stock - (reorder_pt * 2)
        if surplus < min_surplus:
            continue

        # find a sibling that actually needs stock
        needy_siblings = [
            s for s in siblings
            if stock_state.get(s, {}).get(prod_id, 0) < reorder_pt
        ]
        if not needy_siblings:
            continue

        dest_id = random.choice(needy_siblings)
        qty = int(surplus * transfer_frac)
        if qty <= 0:
            continue

        stock_state[node_id][prod_id] -= qty
        stock_state[dest_id][prod_id] += qty

        transfer_ts = datetime.combine(
            current_date,
            datetime.min.time().replace(
                hour=random.randint(10, 15),
                minute=random.randint(0, 59)
            )
        )

        transfers.append({
            "event_id": f"EVT_{event_counter:07d}",
            "event_type": "lateral_transfer",
            "event_timestamp": transfer_ts,
            "node_id": node_id,
            "device_id": pick_scanner(node_id, "outbound"),
            "product_id": prod_id,
            "units_sold":0,
            "units_rto": 0,
            "units_transferred": qty,
            "source_node": node_id,
            "destination_node": dest_id,
            "stock_on_hand":stock_state[node_id][prod_id],
            "rto_reason":None,
            "processing_lag_hours": round(random.uniform(0.5, 2.0), 3),
            "session_id": session_id(node_id, current_date, "LAT"),
        })
        event_counter += 1

    return transfers, event_counter, stock_state

print("Lateral transfer function ready.")

Lateral transfer function ready.


### **Main Loop**

In [8]:

while current_date <= SIM_END:
    day_num = (current_date - SIM_START).days + 1

    # progress tracking metrics
    if day_num % 30 == 0:
        print(f"Processing: Day {day_num}/{total_days}, events so far: {len(events): }")

    for _, ds in darkstores.iterrows():
        # Enforce temporal opening gates (skip darkstores not yet open)
        op_since = date.fromisoformat(str(ds["operational_since"]))
        if current_date < op_since:
            continue
        node_id = ds["node_id"]

        ## 1 (a) transfers arrive early morning 
        try:
            t_events, event_counter, stock_state = generate_transfers(
                current_date, node_id, stock_state, event_counter
            )
            events.extend(t_events)

        except Exception as e:
            print(f"TRANSFER ERROR {node_id} {current_date}: {e}")


        ## 1 (b) lateral transfers during off-peak hours
        try:
            l_events, event_counter, stock_state = generate_lateral_transfers(
                current_date, node_id, stock_state, event_counter
            )
            events.extend(l_events)
        except Exception as e:
            print(f"LATERAL TRANSFER ERROR {node_id} {current_date}: {e}")


        
        ## 2. fulfillments and RTOs throughout the day
        try:
            f_events, event_counter, stock_state = generate_fulfillments_and_rtos(
                current_date, ds, stock_state, event_counter
            )
            events.extend(f_events)
        except Exception as e:
            print(f"Fullfillment error {node_id} {current_date} as {e}")

        ## 3. snapshots at scheduled times
        try:
            s_events, event_counter = generate_snapshots(
                current_date, node_id, "darkstore", stock_state, event_counter
            )
            events.extend(s_events)
        except Exception as e:
            print(f"SNAPSHOT ERROR {node_id} {current_date}: {e}")
    current_date += timedelta(days=1)

df_events = pd.DataFrame(events)
print(f"\nComplete. Total events: {len(df_events):,}")
print(df_events["event_type"].value_counts().to_string())

Processing: Day 30/211, events so far:  47973
Processing: Day 60/211, events so far:  101188
Processing: Day 90/211, events so far:  159834
Processing: Day 120/211, events so far:  223180
Processing: Day 150/211, events so far:  290431
Processing: Day 180/211, events so far:  359663
Processing: Day 210/211, events so far:  429238

Complete. Total events: 436,143
event_type
customer_fulfillment    270837
snapshot                 81372
inbound_transfer         69537
rto_return               11846
lateral_transfer          2551


### validate and save

In [ ]:
print("=== Validation ===")
print(f"Total rows: {len(df_events):,}")
print(f"Date range: {df_events['event_timestamp'].min()} to {df_events['event_timestamp'].max()}")
print(f"Null stock_on_hand: {df_events['stock_on_hand'].isna().sum()}")
print(f"Negative stock: {(df_events['stock_on_hand'] < 0).sum()}")
print(f"RTO with no reason: {((df_events['event_type'] == 'rto_return') & (df_events['rto_reason'].isna())).sum()}")
print(f"Nodes covered: {df_events['node_id'].nunique()}")


df_events.to_csv("events_baseline.csv", index = False)

print(f"\nSaved")

=== Validation ===
Total rows: 436,143
Date range: 2025-12-01 06:00:00 to 2026-07-01 05:27:00
Null stock_on_hand: 0
Negative stock: 0
RTO with no reason: 0
Nodes covered: 100

Saved
